In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,BertForSequenceClassification

## load bert model

In [2]:
class NewsClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertForSequenceClassification.from_pretrained("google-bert/bert-base-uncased",num_labels=2)
        for param in self.bert.bert.parameters():
            param.requires_grad=False # freeze all BERT layers
            
        for param in self.bert.bert.encoder.layer[-1].parameters():
            param.requires_grad = True
    def forward(self,input_ids,attention_mask):
        bert_output = self.bert(input_ids=input_ids,attention_mask=attention_mask)
        
        return bert_output.logits



In [3]:
model = NewsClassifier()
model.load_state_dict(torch.load("balanced_news_model.pth"))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<All keys matched successfully>

### Tokenization to convert predict text into tokens that will be pass into model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
def predict(text, model=model, tokenizer=tokenizer):
    model.eval()

    encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']

    with torch.no_grad():
        logits = model(input_ids, attention_mask)

    pred = torch.argmax(logits, dim=1)

    return pred.item()

### prediction

In [28]:
text = "Iran says it fired missile at USS Abraham Lincoln"
predict(text)

0